# Sentinel: fine-tune the fire/smoke detector (YOLOX-Tiny on D-Fire), cloud GPU

**Ran on Kaggle (T4) on 2026-10-14: 80 epochs, ~4 h 39 min. Results: `ai-service/reports/README.md`.**

Runs on **Google Colab** (Runtime → Change runtime type → T4 GPU) or **Kaggle** (Notebook → Settings → Accelerator GPU T4 x2, Internet ON; upload `D-Fire.zip` and `ai-service.zip` as private Kaggle Datasets and Add them to the notebook). Kaggle free tier: ~30 GPU hours/week, 12 h per session.

Inputs you upload to Google Drive (folder `MyDrive/sentinel/`):
1. `D-Fire.zip`, the official download from https://github.com/gaiasd/DFireDataset (CC0 1.0; please cite the authors).
2. `ai-service.zip`, a zip of this repo's `ai-service/` folder (only the code and `models/models.json` are needed).

Outputs written to `MyDrive/sentinel/out/`:
- `fire_smoke_yolox.onnx` and its sha256
- `best_ckpt.pth`
- the training log
- the evaluation report
- `pip freeze`

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU: switch the runtime to GPU first'
print('torch', torch.__version__, 'cuda', torch.version.cuda)

In [ ]:
import os, glob, shutil, subprocess
KAGGLE = os.path.isdir('/kaggle/input')
if KAGGLE:  # inputs = your uploaded Kaggle datasets (auto-unzipped); outputs = /kaggle/working (download before the session ends)
    SRC, DRIVE = '/kaggle/input', '/kaggle/working/sentinel'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    SRC = DRIVE = '/content/drive/MyDrive/sentinel'
    assert os.path.isdir(DRIVE), 'MyDrive/sentinel folder nahi mila (naam chhote "s" se rakhein)'
print('KAGGLE' if KAGGLE else 'COLAB', '| inputs in', SRC, ':', sorted(os.listdir(SRC)))
os.makedirs(f'{DRIVE}/out', exist_ok=True); os.makedirs('/content/work', exist_ok=True)

# ai-service: zip OR an unzipped folder (Windows "Extract All" may nest it one level deeper)
if os.path.isfile(f'{SRC}/ai-service.zip'):
    subprocess.run(['unzip', '-q', '-o', f'{SRC}/ai-service.zip', '-d', '/content/work'], check=True)
else:
    found = glob.glob(f'{SRC}/**/tools/prepare_dataset.py', recursive=True)
    assert found, 'ai-service nahi mila: ai-service.zip ya ai-service folder sentinel mein upload karein'
    shutil.copytree(os.path.dirname(os.path.dirname(found[0])), '/content/work/ai-service', dirs_exist_ok=True)

# D-Fire: zip OR an unzipped folder that contains train/images and test/images
if os.path.isfile(f'{SRC}/D-Fire.zip'):
    size = os.path.getsize(f'{SRC}/D-Fire.zip')
    assert size == 3036222313, f'D-Fire.zip {size} bytes hai, 3036222313 hona chahiye: upload adhoora hai'
    subprocess.run(['unzip', '-q', '-o', f'{SRC}/D-Fire.zip', '-d', '/content/work/dfire'], check=True)
else:
    found = [os.path.dirname(os.path.dirname(t)) for t in glob.glob(f'{SRC}/**/train/images', recursive=True)]
    found = [d for d in found if os.path.isdir(f'{d}/test/images')]
    assert found, 'D-Fire nahi mila: sentinel mein D-Fire.zip, ya woh folder jiske andar train/ aur test/ hon, upload karein'
    print('copying D-Fire from Drive to local disk (10-30 min for ~21k files)...', found[0])
    shutil.copytree(found[0], '/content/work/dfire', dirs_exist_ok=True)

# Official D-Fire counts: a smaller number means the Drive upload skipped files.
for split, n in (('train', 17221), ('test', 4306)):
    imgs = len(os.listdir(f'/content/work/dfire/{split}/images')); labs = len(os.listdir(f'/content/work/dfire/{split}/labels'))
    print(split, imgs, 'images,', labs, 'labels')
    assert imgs == n and labs == n, f'{split}: {imgs} images / {labs} labels, {n} hone chahiye: Drive upload adhoora hai, missing files dobara upload karein (ya D-Fire.zip upload karein)'
print('OK: data complete', os.listdir('/content/work/ai-service'))


## 1. Validate the official train and test splits

Adjust `TRAIN` and `TEST` if the printed folder structure differs. Each split needs `images/` and `labels/` folders. **Never** use test images for training.

In [ ]:
import glob
TRAIN, TEST = '/content/work/dfire/train', '/content/work/dfire/test'
assert os.path.isdir(f'{TRAIN}/images') and os.path.isdir(f'{TEST}/images'), f'train/test nahi mile, unzip ke baad mila: {glob.glob("/content/work/dfire/*")}'
print(TRAIN, len(os.listdir(f'{TRAIN}/images')), 'images |', TEST, len(os.listdir(f'{TEST}/images')), 'images')
!cd /content/work/ai-service && python tools/prepare_dataset.py validate {TRAIN} --classes 2
!cd /content/work/ai-service && python tools/prepare_dataset.py validate {TEST} --classes 2


## 2. Carve a validation set out of TRAIN, grouped by source

Frames from one source stay in one split. The group key is the filename without its last two digits, so blocks of 100 consecutive frames stay together (for example `AoF06723.jpg` → group `AoF067`). Then build the COCO json that YOLOX needs. Class ids: 1 = smoke, 2 = fire, so YOLOX index 0 = smoke and 1 = fire, matching `models.json`.

In [ ]:
!cd /content/work/ai-service && python tools/prepare_dataset.py split {TRAIN} /content/work/split --val 0.15 --test 0 --group '^(.*)\d\d\.\w+$'
!cd /content/work/ai-service && python tools/prepare_dataset.py coco /content/work/split/train /content/work/dfire_coco/annotations/train.json --names smoke,fire
!cd /content/work/ai-service && python tools/prepare_dataset.py coco /content/work/split/val /content/work/dfire_coco/annotations/val.json --names smoke,fire
!mkdir -p /content/work/dfire_coco && rm -rf /content/work/dfire_coco/train2017 /content/work/dfire_coco/val2017
!cp -r /content/work/split/train/images /content/work/dfire_coco/train2017 && cp -r /content/work/split/val/images /content/work/dfire_coco/val2017
!ls /content/work/dfire_coco/train2017 | wc -l; ls /content/work/dfire_coco/val2017 | wc -l

## 3. Install YOLOX (Apache-2.0) and the official COCO start checkpoint

The `.pth` file is a PyTorch checkpoint from the official Megvii release. PyTorch 2.6 and later load it in tensors-only mode (`weights_only=True`), so no code inside it runs. Its sha256 is printed and saved for the record (Kaggle run 2026-10-14: 9de513de589ac98bb92d3bca53b5af7b9acfa9b0bacb831f7999d0f7afaee8f0).

In [ ]:
# YOLOX is used from its folder via PYTHONPATH: `pip install -e .` fails because its old setup.py needs torch at build time.
%env PYTHONPATH=/content/work/YOLOX
import os
if not os.path.isdir('/content/work/YOLOX'):
    !cd /content/work && git clone -q --branch 0.3.0 https://github.com/Megvii-BaseDetection/YOLOX
!pip -q install loguru tabulate pycocotools thop ninja onnx
if not os.path.isfile('/content/work/YOLOX/yolox_tiny.pth'):
    !cd /content/work/YOLOX && wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_tiny.pth
!sha256sum /content/work/YOLOX/yolox_tiny.pth | tee {DRIVE}/out/yolox_tiny_pth.sha256
!cd /content/work/YOLOX && python -c "import yolox; print('YOLOX OK', yolox.__version__)"


## 4. Fine-tune

Measured on Kaggle T4 (2026-10-14): ~0.45 s/iteration, ~3.5 min/epoch, so 80 epochs take **~5 h** plus evaluations. Use Save & Run All (Commit) so it keeps running with the browser closed. `best_ckpt.pth` is the checkpoint with the best validation AP.

**If the GPU session ends midway:** open the notebook again, run every cell from the top (the data steps repeat the same split, seed 42), and this cell continues from `latest_ckpt.pth` in `sentinel/out/YOLOX_outputs` (Colab: on Drive; Kaggle: download `/kaggle/working/sentinel/out` before the session ends).


In [ ]:
%env DFIRE_COCO=/content/work/dfire_coco
OUTD = f'{DRIVE}/out/YOLOX_outputs'
CKPT = f'{OUTD}/yolox_tiny_dfire/latest_ckpt.pth'
START = f'--resume -c {CKPT}' if os.path.isfile(CKPT) else '-c yolox_tiny.pth'
print('RESUMING from', CKPT) if os.path.isfile(CKPT) else print('starting from the official COCO yolox_tiny.pth')
!cd /content/work/YOLOX && python tools/train.py -f /content/work/ai-service/training/yolox_tiny_dfire.py -d 1 -b 32 --fp16 -o {START} output_dir {OUTD} 2>&1 | tee -a {DRIVE}/out/train.log


## 5. Export ONNX (raw head output; `sentinel_ai` decodes it, same as the person model)

In [ ]:
# Export with training/export_onnx.py (YOLOX 0.3.0's own exporter uses torch.onnx._export, removed from current PyTorch).
# A shell (!) call never raises, so a failed export cannot throw away the checkpoints (Kaggle keeps no output of a FAILED run).
import os
ONNX = f'{DRIVE}/out/fire_smoke_yolox.onnx'
!python /content/work/ai-service/training/export_onnx.py --exp /content/work/ai-service/training/yolox_tiny_dfire.py --ckpt {OUTD}/yolox_tiny_dfire/best_ckpt.pth --out {ONNX}
if os.path.isfile(ONNX):
    !sha256sum {ONNX} | tee {DRIVE}/out/fire_smoke_yolox.onnx.sha256
else:
    print('!!! EXPORT FAILED, but best_ckpt.pth is saved in', OUTD, '(export it locally with training/export_onnx.py)')
!pip freeze > {DRIVE}/out/pip-freeze.txt


## 6. Evaluate on the held-out official TEST split, with the same tool used for the person model

The report covers fire, smoke and negative images separately. It includes TP/FP/FN, precision, recall and F1, an image-level confusion matrix, the false-positive rate, the hard-case slices (smoke only, small fire), the failure files and latency (note that Colab latency is not your deployment CPU latency).

In [ ]:
import json
if not os.path.isfile(ONNX):
    print('!!! no ONNX model, evaluation skipped (the run still completes so the checkpoints are kept)')
else:
    digest = open(f'{DRIVE}/out/fire_smoke_yolox.onnx.sha256').read().split()[0]
    !cp {ONNX} /content/work/ai-service/models/
    m = json.load(open('/content/work/ai-service/models/models.json'))
    for e in m['models']:
        if e['name'] == 'fire_smoke': e['sha256'] = digest
    json.dump(m, open('/content/work/ai-service/models/models.json', 'w'), indent=2)
    !pip -q install onnxruntime opencv-python-headless
    !cd /content/work/ai-service && python tools/evaluate.py --data {TEST} --classes 0=Smoke,1=Fire --sweep 0.25,0.3,0.35,0.4,0.5 --out {DRIVE}/out/eval-dfire-test || echo '!!! evaluation failed, see above'


## 7. Bring the model home
1. Copy `out/fire_smoke_yolox.onnx` to `ai-service/models/`.
2. Put the sha256 into `models/models.json` (entry `fire_smoke`, field `sha256`).
3. Run locally: `python tools/evaluate.py --data <D-Fire test> --classes 0=Smoke,1=Fire --out reports/fire-local` to measure CPU latency on the real deployment machine.
4. Record the decision in `ai-service/reports/README.md`.